# Report Charts: Base Model, SFT, KTO, KTO+SFT

Notebook này tạo các ảnh/bảng báo cáo:

- Loss của `SFT`, `KTO`, `KTO+SFT`.
- Auto metrics trên tập test: Token F1, ROUGE-L, BLEU, PhoBERT Semantic Similarity.
- Signal từ LLM-as-judge: `chosen`, `reject`, `neutral`.
- Taxonomy riêng cho từng model, chỉ tính `CLASS_1` và `CLASS_2`.

Chạy notebook từ repo root hoặc từ thư mục `notebooks/` đều được. Chỉ cần sửa block cấu hình bên dưới nếu đổi thư mục kết quả.

In [ ]:
from pathlib import Path
import json
import sys
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.cli.build_judged_kto_data import decide_binary_label, extract_json_object, triggered_or_violated_labels

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

In [ ]:
# ===== CẤU HÌNH DỄ SỬA =====

SUITE_DIR = PROJECT_ROOT / "results" / "model_suite_report_test"
OUTPUT_DIR = PROJECT_ROOT / "results" / "report_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = [
    {"slug": "base_model", "name": "Base Model", "aliases": ["base_model", "qwen3_8b", "qwen3-8b"]},
    {"slug": "sft", "name": "SFT", "aliases": ["sft", "sft_final"]},
    {"slug": "kto", "name": "KTO", "aliases": ["kto", "kto_base", "kto_base_violation_pair"]},
    {"slug": "kto_sft", "name": "KTO+SFT", "aliases": ["kto_sft", "sft_kto", "kto_violation_pair"]},
]

TRAIN_DIRS = {
    "SFT": PROJECT_ROOT / "models" / "sft_checkpoints",
    "KTO": PROJECT_ROOT / "models" / "kto_base_violation_pair_checkpoints",
    "KTO+SFT": PROJECT_ROOT / "models" / "kto_violation_pair_checkpoints",
}

METRIC_LABELS = {
    "token_f1": "Token F1",
    "rouge_l": "ROUGE-L",
    "bleu": "BLEU",
    "phobert_semantic_similarity": "PhoBERT Similarity",
}

SIGNAL_ORDER = ["chosen", "reject", "neutral"]
TAXONOMY_CLASSES = {"CLASS_1", "CLASS_2"}
TOP_TAXONOMY_LABELS = 30

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SUITE_DIR =", SUITE_DIR)
print("OUTPUT_DIR =", OUTPUT_DIR)

## Helper Functions

In [ ]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def model_dir(spec: dict) -> Path | None:
    for alias in spec["aliases"]:
        candidate = SUITE_DIR / alias
        if candidate.exists():
            return candidate
    return None


def resolve_trainer_state_path(train_dir: Path) -> Path | None:
    direct = train_dir / "trainer_state.json"
    if direct.exists():
        return direct

    def checkpoint_step(path: Path) -> int:
        try:
            return int(path.parent.name.rsplit("-", maxsplit=1)[1])
        except (IndexError, ValueError):
            return -1

    candidates = sorted(train_dir.glob("checkpoint-*/trainer_state.json"), key=checkpoint_step)
    return candidates[-1] if candidates else None


def load_loss_rows(train_dirs: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for run_name, train_dir in train_dirs.items():
        state_path = resolve_trainer_state_path(train_dir)
        if state_path is None:
            print(f"Missing trainer_state.json for {run_name}: {train_dir}")
            continue
        state = read_json(state_path)
        for item in state.get("log_history", []):
            step = item.get("step")
            if step is None:
                continue
            if "loss" in item:
                rows.append({"run": run_name, "step": step, "metric": "train_loss", "value": item["loss"]})
            if "eval_loss" in item:
                rows.append({"run": run_name, "step": step, "metric": "eval_loss", "value": item["eval_loss"]})
    return pd.DataFrame(rows)


def load_autometrics_rows() -> pd.DataFrame:
    rows = []
    for spec in MODEL_SPECS:
        directory = model_dir(spec)
        if directory is None:
            print(f"Missing suite dir for {spec['name']}")
            continue
        summary_path = directory / "autometrics" / "autometrics_summary.json"
        if not summary_path.exists():
            print(f"Missing autometrics for {spec['name']}: {summary_path}")
            continue
        summary = read_json(summary_path)
        row = {"model": spec["name"], "slug": spec["slug"], "num_samples": summary.get("num_samples", 0)}
        for metric, stats in summary.get("metrics", {}).items():
            row[f"{metric}_mean"] = stats.get("mean")
            row[f"{metric}_median"] = stats.get("median")
            row[f"{metric}_std"] = stats.get("std")
        rows.append(row)
    return pd.DataFrame(rows)


def load_judge_records() -> dict[str, list[dict]]:
    records_by_model = {}
    for spec in MODEL_SPECS:
        directory = model_dir(spec)
        if directory is None:
            print(f"Missing suite dir for {spec['name']}")
            records_by_model[spec["name"]] = []
            continue
        path = directory / "judge" / "evaluation_results.json"
        if not path.exists():
            print(f"Missing judge results for {spec['name']}: {path}")
            records_by_model[spec["name"]] = []
            continue
        records_by_model[spec["name"]] = read_json(path)
    return records_by_model


def signal_from_record(record: dict) -> str:
    try:
        label, decision, _reason, _active_labels = decide_binary_label(record)
    except Exception:
        return "neutral"
    if label is True:
        return "chosen"
    if label is False:
        return "reject"
    return "neutral"


def collect_signal_counts(records_by_model: dict[str, list[dict]]) -> pd.DataFrame:
    rows = []
    for model_name, records in records_by_model.items():
        counts = Counter(signal_from_record(record) for record in records)
        for signal in SIGNAL_ORDER:
            rows.append({"model": model_name, "signal": signal, "count": counts.get(signal, 0)})
    return pd.DataFrame(rows)


def collect_taxonomy_counts(records_by_model: dict[str, list[dict]]) -> pd.DataFrame:
    rows = []
    for model_name, records in records_by_model.items():
        for record in records:
            class_label = str(record.get("judge_classification_label") or "UNKNOWN")
            if class_label not in TAXONOMY_CLASSES:
                continue
            payload = extract_json_object(record.get("judge_evaluation"))
            if not payload:
                labels = ["parse_error_or_empty"]
            else:
                labels = triggered_or_violated_labels(payload) or ["no_triggered_taxonomy"]
            for label in labels:
                rows.append({"model": model_name, "class": class_label, "taxonomy": label, "count": 1})
    if not rows:
        return pd.DataFrame(columns=["model", "class", "taxonomy", "count"])
    return pd.DataFrame(rows).groupby(["model", "class", "taxonomy"], as_index=False)["count"].sum()


def save_table(df: pd.DataFrame, stem: str) -> None:
    df.to_csv(OUTPUT_DIR / f"{stem}.csv", index=False, encoding="utf-8")
    (OUTPUT_DIR / f"{stem}.md").write_text(df.to_markdown(index=False), encoding="utf-8")

## 1. Loss Chart: SFT, KTO, KTO+SFT

In [ ]:
loss_df = load_loss_rows(TRAIN_DIRS)
save_table(loss_df, "training_loss")
loss_df.head()

In [ ]:
if loss_df.empty:
    print("No loss data found.")
else:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for (run, metric), group in loss_df.groupby(["run", "metric"]):
        group = group.sort_values("step")
        linestyle = "-" if metric == "train_loss" else "--"
        marker = "o" if metric == "eval_loss" else None
        ax.plot(group["step"], group["value"], linestyle=linestyle, marker=marker, linewidth=1.8, label=f"{run} {metric}")
    ax.set_title("Training/Evaluation Loss")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.legend(ncol=2)
    fig.tight_layout()
    path = OUTPUT_DIR / "loss_sft_kto_kto_sft.png"
    fig.savefig(path)
    print(path)

## 2. Auto Metrics Table + Chart trên tập test

Cần có file `autometrics/autometrics_summary.json` trong từng thư mục model của `SUITE_DIR`.

In [ ]:
autometrics_df = load_autometrics_rows()
save_table(autometrics_df, "auto_metrics_by_model")
autometrics_df

In [ ]:
mean_columns = [column for column in autometrics_df.columns if column.endswith("_mean")]
if autometrics_df.empty or not mean_columns:
    print("No autometrics data found.")
else:
    plot_rows = []
    for _, row in autometrics_df.iterrows():
        for column in mean_columns:
            metric = column.removesuffix("_mean")
            plot_rows.append({
                "model": row["model"],
                "metric": METRIC_LABELS.get(metric, metric),
                "value": row[column],
            })
    plot_df = pd.DataFrame(plot_rows).dropna()
    pivot = plot_df.pivot(index="metric", columns="model", values="value")
    pivot = pivot[[spec["name"] for spec in MODEL_SPECS if spec["name"] in pivot.columns]]
    ax = pivot.plot(kind="bar", figsize=(11, 5.5), rot=0, width=0.8)
    ax.set_title("Auto Metrics on Test Set")
    ax.set_xlabel("")
    ax.set_ylabel("Mean score")
    ax.set_ylim(0, 1)
    ax.legend(title="Model")
    fig = ax.get_figure()
    fig.tight_layout()
    path = OUTPUT_DIR / "auto_metrics_by_model.png"
    fig.savefig(path)
    print(path)

## 3. LLM-as-Judge Signal: chosen / reject / neutral

Mapping:

- `chosen`: judge decision tương ứng KTO positive.
- `reject`: judge decision tương ứng KTO negative.
- `neutral`: `CLASS_3`, lỗi parse, hoặc decision bị drop/không đủ mạnh để dùng làm binary signal.

In [ ]:
records_by_model = load_judge_records()
signal_df = collect_signal_counts(records_by_model)
save_table(signal_df, "judge_signal_counts")
signal_df

In [ ]:
if signal_df.empty:
    print("No judge signal data found.")
else:
    pivot = signal_df.pivot_table(index="model", columns="signal", values="count", aggfunc="sum", fill_value=0)
    pivot = pivot.reindex([spec["name"] for spec in MODEL_SPECS]).fillna(0)
    pivot = pivot[[signal for signal in SIGNAL_ORDER if signal in pivot.columns]]
    ax = pivot.plot(kind="bar", stacked=True, figsize=(10, 5.5), rot=0)
    ax.set_title("LLM-as-Judge Signal Counts")
    ax.set_xlabel("")
    ax.set_ylabel("Samples")
    ax.legend(title="Signal")
    fig = ax.get_figure()
    fig.tight_layout()
    path = OUTPUT_DIR / "judge_signal_chosen_reject_neutral.png"
    fig.savefig(path)
    print(path)

## 4. Taxonomy Charts riêng từng model, chỉ CLASS_1 và CLASS_2

Notebook cố ý bỏ `CLASS_3` khỏi taxonomy chart theo yêu cầu report mới.

In [ ]:
taxonomy_df = collect_taxonomy_counts(records_by_model)
save_table(taxonomy_df, "taxonomy_counts_class1_class2")
taxonomy_df.head(20)

In [ ]:
if taxonomy_df.empty:
    print("No taxonomy data found.")
else:
    for spec in MODEL_SPECS:
        model_name = spec["name"]
        model_tax = taxonomy_df[taxonomy_df["model"] == model_name].copy()
        if model_tax.empty:
            print(f"No taxonomy data for {model_name}")
            continue
        top_labels = (
            model_tax.groupby("taxonomy")["count"]
            .sum()
            .sort_values(ascending=False)
            .head(TOP_TAXONOMY_LABELS)
            .index
        )
        model_tax = model_tax[model_tax["taxonomy"].isin(top_labels)]
        pivot = model_tax.pivot_table(index="taxonomy", columns="class", values="count", aggfunc="sum", fill_value=0)
        pivot["total"] = pivot.sum(axis=1)
        pivot = pivot.sort_values("total", ascending=True).drop(columns=["total"])

        ax = pivot.plot(kind="barh", stacked=True, figsize=(11, max(5, 0.35 * len(pivot))))
        ax.set_title(f"Taxonomy Counts - {model_name} (CLASS_1/CLASS_2 only)")
        ax.set_xlabel("Count")
        ax.set_ylabel("")
        ax.legend(title="Class")
        fig = ax.get_figure()
        fig.tight_layout()
        path = OUTPUT_DIR / f"taxonomy_{spec['slug']}_class1_class2.png"
        fig.savefig(path)
        print(path)

## Files Generated

In [ ]:
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.relative_to(PROJECT_ROOT))

## Gợi ý chạy eval suite trước khi vẽ đủ 4 model

Nếu `SUITE_DIR` chưa có đủ 4 model, chạy trên server bằng Slurm script đã thêm:

```bash
sbatch --export=ALL,EVAL_NUM_SAMPLES=1971,SUITE_DIR=/data2/cmdir/home/ioit107/vqhuy/rl-for-llm-edu/results/model_suite_report_test scripts/slurm/13_eval_report_suite.sbatch
```

Smoke test nhanh:

```bash
sbatch --export=ALL,EVAL_NUM_SAMPLES=48,SUITE_DIR=/data2/cmdir/home/ioit107/vqhuy/rl-for-llm-edu/results/model_suite_report_smoke scripts/slurm/13_eval_report_suite.sbatch
```